# HELM — Training History & Model Comparison

Visualize training progress and compare method variants.

- **Section 1**: Training & validation loss curves for all available models
- **Section 2**: Individual loss components for the full HELM model (cls, graph, BYOL)

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "legend.frameon": True,
    "legend.edgecolor": "0.8",
    "legend.facecolor": "white",
})

## Configuration

In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────
DATASET = "ucm"
FRACTION = 100            # labeled fraction (e.g., 1, 5, 10, 25, 100)
SEED = 42

LOGS_DIR = "../logs"      # relative to notebooks/

# Methods to compare (order = plot order)
METHODS = [
    "mlc-sl",
    "hmlc-sl",
    "hmlc-sl-graph",
    "hmlc-sl-graph-byol",
    "hmlc-ssl-byol",
    "hmlc-ssl-graph",
    "hmlc-ssl-graph-byol",
]

METHOD_LABELS = {
    "mlc-sl": "MLC (flat baseline)",
    "hmlc-sl": "HMLC",
    "hmlc-sl-graph": "HMLC + Graph",
    "hmlc-sl-graph-byol": "HMLC + Graph + BYOL",
    "hmlc-ssl-byol": "HMLC + BYOL (SSL)",
    "hmlc-ssl-graph": "HMLC + Graph (SSL)",
    "hmlc-ssl-graph-byol": "HELM (full)",
}

METHOD_COLORS = {
    "mlc-sl": "#999999",
    "hmlc-sl": "#E8871E",
    "hmlc-sl-graph": "#57A773",
    "hmlc-sl-graph-byol": "#2E86AB",
    "hmlc-ssl-byol": "#9B59B6",
    "hmlc-ssl-graph": "#1ABC9C",
    "hmlc-ssl-graph-byol": "#E74C3C",
}

# Full HELM method name (for individual loss breakdown)
HELM_METHOD = "hmlc-ssl-graph-byol"

## Load logs

In [ ]:
def load_metrics(dataset, method, fraction, seed, logs_dir=LOGS_DIR):
    """Load metrics.csv for a given run. Returns None if not found."""
    csv_path = os.path.join(
        logs_dir, dataset, method, f"fraction_{fraction}",
        f"seed_{seed}", "version_0", "metrics.csv",
    )
    if os.path.exists(csv_path):
        return pd.read_csv(csv_path)
    return None


# Discover all available logs
available = {}
for method in METHODS:
    df = load_metrics(DATASET, method, FRACTION, SEED)
    if df is not None:
        available[method] = df
        print(f"  Found: {METHOD_LABELS.get(method, method):30s} ({len(df)} rows)")

if not available:
    print(f"\nNo logs found in {LOGS_DIR}/{DATASET}/*/fraction_{FRACTION}/seed_{SEED}/")
    print("Train some models first, then re-run this notebook.")
else:
    print(f"\n{len(available)} model(s) found for {DATASET}, fraction {FRACTION}%, seed {SEED}.")

## Section 1 — Training & Validation Loss Comparison

Per-step train loss and per-epoch validation loss for every available method, overlaid for direct comparison.

In [ ]:
if available:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Train loss (per step) ────────────────────────────────────────────
    ax = axes[0]
    for method, df in available.items():
        train = df.dropna(subset=["train_loss_step"])
        ax.plot(
            range(len(train)), train["train_loss_step"].values,
            color=METHOD_COLORS.get(method, "#333"),
            linewidth=1.2, alpha=0.85,
            label=METHOD_LABELS.get(method, method),
        )
    ax.set_xlabel("Training step")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss (per step)")
    ax.legend(loc="upper right")

    # ── Validation loss (per epoch) ──────────────────────────────────────
    ax = axes[1]
    for method, df in available.items():
        val = df.dropna(subset=["val_loss_epoch"])
        epochs = val["epoch"].values
        ax.plot(
            epochs, val["val_loss_epoch"].values,
            color=METHOD_COLORS.get(method, "#333"),
            linewidth=1.8, marker="o", markersize=4,
            markeredgecolor="white", markeredgewidth=0.5,
            label=METHOD_LABELS.get(method, method),
        )
        # Mark best epoch
        best_idx = val["val_loss_epoch"].idxmin()
        best_epoch = val.loc[best_idx, "epoch"]
        best_val = val.loc[best_idx, "val_loss_epoch"]
        ax.plot(best_epoch, best_val, marker="*", markersize=12,
                color=METHOD_COLORS.get(method, "#333"),
                markeredgecolor="white", markeredgewidth=0.5, zorder=5)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Validation Loss (per epoch)")
    ax.legend(loc="upper right")

    fig.suptitle(
        f"Training Progress — {DATASET.upper()}, {FRACTION}% labeled, seed {SEED}",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    plt.show()
else:
    print("No logs to plot.")

### Per-model train vs validation loss

One subplot per method, showing train (per step) and validation (per epoch) on the same axes.

In [ ]:
if available:
    n = len(available)
    n_cols = min(3, n)
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 4 * n_rows), squeeze=False)

    for idx, (method, df) in enumerate(available.items()):
        row, col = divmod(idx, n_cols)
        ax = axes[row][col]
        color = METHOD_COLORS.get(method, "#333")

        # Train loss per step
        train = df.dropna(subset=["train_loss_step"])
        steps = range(len(train))
        ax.plot(steps, train["train_loss_step"].values,
                color=color, linewidth=0.8, alpha=0.7, label="Train (step)")

        # Validation loss per epoch — plot at the step position of each epoch boundary
        val = df.dropna(subset=["val_loss_epoch"])
        if len(val) > 0 and len(train) > 0:
            epochs = val["epoch"].values
            n_steps = len(train)
            max_epoch = epochs.max() + 1 if len(epochs) > 0 else 1
            step_positions = (epochs / max_epoch * n_steps).astype(int)
            ax.plot(step_positions, val["val_loss_epoch"].values,
                    color=color, linewidth=2.0, marker="o", markersize=5,
                    markeredgecolor="white", markeredgewidth=0.6,
                    label="Val (epoch)")

            # Best epoch star
            best_i = val["val_loss_epoch"].idxmin()
            best_ep = val.loc[best_i, "epoch"]
            best_step = int(best_ep / max_epoch * n_steps)
            best_val = val.loc[best_i, "val_loss_epoch"]
            ax.plot(best_step, best_val, marker="*", markersize=14,
                    color=color, markeredgecolor="white", markeredgewidth=0.6, zorder=5)

        ax.set_title(METHOD_LABELS.get(method, method), color=color)
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.legend(loc="upper right", fontsize=8)

    # Hide unused subplots
    for idx in range(n, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row][col].set_visible(False)

    fig.suptitle(
        f"Train vs Validation Loss — {DATASET.upper()}, {FRACTION}% labeled",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    plt.show()

## Section 2 — HELM Individual Loss Components

Breakdown of the full HELM model's loss into classification (BCE), graph (GraphSAGE), and BYOL (contrastive) components. Only available if `hmlc-ssl-graph-byol` logs exist with individual loss columns.

In [ ]:
LOSS_COMPONENTS = {
    "train_loss_cls":  {"label": "Classification (BCE)",  "color": "#E74C3C"},
    "train_loss_gcn":  {"label": "Graph (SAGE)",          "color": "#2E86AB"},
    "train_loss_byol": {"label": "BYOL (Contrastive)",    "color": "#57A773"},
}

helm_df = available.get(HELM_METHOD)

if helm_df is not None:
    # Check which individual loss columns are present
    found_cols = [c for c in LOSS_COMPONENTS if c in helm_df.columns]

    if found_cols:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # ── Left: Individual losses per step ─────────────────────────────
        ax = axes[0]
        for col in found_cols:
            series = helm_df.dropna(subset=[col])
            cfg = LOSS_COMPONENTS[col]
            ax.plot(
                range(len(series)), series[col].values,
                color=cfg["color"], linewidth=1.0, alpha=0.85,
                label=cfg["label"],
            )
        ax.set_xlabel("Training step")
        ax.set_ylabel("Loss")
        ax.set_title("Individual Loss Components (per step)")
        ax.legend(loc="upper right")

        # ── Right: Stacked area showing relative contribution ────────────
        ax = axes[1]
        # Align all columns to same steps
        step_df = helm_df.dropna(subset=found_cols)[found_cols].reset_index(drop=True)
        if len(step_df) > 0:
            steps = range(len(step_df))
            bottom = np.zeros(len(step_df))
            for col in found_cols:
                cfg = LOSS_COMPONENTS[col]
                vals = step_df[col].values
                ax.fill_between(steps, bottom, bottom + vals,
                                color=cfg["color"], alpha=0.6,
                                label=cfg["label"])
                bottom += vals
            ax.set_xlabel("Training step")
            ax.set_ylabel("Cumulative loss")
            ax.set_title("Loss Composition (stacked)")
            ax.legend(loc="upper right")

        fig.suptitle(
            f"HELM Loss Breakdown — {DATASET.upper()}, {FRACTION}% labeled, seed {SEED}",
            fontsize=13, fontweight="bold", y=1.02,
        )
        plt.tight_layout()
        plt.show()

        # ── Per-component subplots ───────────────────────────────────────
        n_comp = len(found_cols)
        fig, axes = plt.subplots(1, n_comp, figsize=(5 * n_comp, 4))
        if n_comp == 1:
            axes = [axes]

        for i, col in enumerate(found_cols):
            ax = axes[i]
            cfg = LOSS_COMPONENTS[col]
            series = helm_df.dropna(subset=[col])
            ax.plot(range(len(series)), series[col].values,
                    color=cfg["color"], linewidth=1.0)
            ax.set_xlabel("Training step")
            ax.set_ylabel("Loss")
            ax.set_title(cfg["label"], color=cfg["color"])

        fig.suptitle(
            f"HELM Individual Losses — {DATASET.upper()}, {FRACTION}% labeled",
            fontsize=13, fontweight="bold", y=1.02,
        )
        plt.tight_layout()
        plt.show()

    else:
        print(f"HELM logs found but no individual loss columns (train_loss_cls, train_loss_gcn, train_loss_byol).")
        print("These are logged by the updated ssl_graph_byol trainer. Re-train to generate them.")
        print(f"\nAvailable columns: {list(helm_df.columns)}")
else:
    print(f"No logs found for {HELM_METHOD}. Train it first:")
    print(f"  python train.py dataset={DATASET} method={HELM_METHOD} experiment.seeds=[{SEED}]")